# DuckDB Spatial Analytics Layer


## Purpose



The database integrates the processed Silver-layer datasets and prepares reusable analytical tables for downstream Gold-layer analysis and Power BI.

The workflow in this notebook includes:

- creating a persistent DuckDB database;
- enabling spatial functionality;
- validating spatial SQL functions;
- loading Silver GeoParquet datasets;
- inspecting database schemas and spatial geometry types;
- preparing the data for analytical modelling.

Pipeline:

EV Silver
+
ABS SA4 Silver
+
EV-SA4 Enriched Silver
↓
DuckDB
↓
Spatial Extension
↓
Validated Analytical Tables
↓
Gold Analytics
↓
Power BI

The database integrates the processed Silver-layer datasets and prepares reusable analytical tables for downstream Gold-layer analysis and Power BI.

The workflow in this notebook includes:

- creating a persistent DuckDB database;
- enabling spatial functionality;
- validating spatial SQL functions;
- loading Silver GeoParquet datasets;
- inspecting database schemas and spatial geometry types;
- preparing the data for analytical modelling.

Pipeline:

EV Silver
+
ABS SA4 Silver
+
EV-SA4 Enriched Silver
↓
DuckDB
↓
Spatial Extension
↓
Validated Analytical Tables
↓
Gold Analytics
↓
Power BI

## Step 1 — Configure Database and Silver Data Paths



The project paths are configured for the DuckDB database and the three processed Silver datasets.

The analytical database is stored separately from the raw and processed files so that the pipeline maintains clear separation between:

- Bronze source data;
- Silver cleaned and spatially enriched data;
- database and analytical outputs.

The three Silver inputs are:

1. cleaned EV charging locations;
2. NSW SA4 polygon boundaries;
3. EV charging locations enriched with SA4 attributes.

In [4]:
from pathlib import Path

import duckdb


PROJECT_ROOT = Path.cwd().parent

DATABASE_DIR = PROJECT_ROOT / "data" / "database"
DATABASE_DIR.mkdir(parents=True, exist_ok=True)

DATABASE_PATH = DATABASE_DIR / "nsw_ev.duckdb"

EV_SILVER_PATH = (
    PROJECT_ROOT
    / "data"
    / "silver"
    / "ev"
    / "ev_charging_locations_silver.parquet"
)

SA4_SILVER_PATH = (
    PROJECT_ROOT
    / "data"
    / "silver"
    / "abs"
    / "nsw_sa4_silver.parquet"
)

EV_SA4_SILVER_PATH = (
    PROJECT_ROOT
    / "data"
    / "silver"
    / "ev_sa4"
    / "ev_charging_locations_sa4_silver.parquet"
)

print("DuckDB version:", duckdb.__version__)
print("Project root:", PROJECT_ROOT)
print("Database:", DATABASE_PATH)

print("\nSilver files:")
print("EV Silver:", EV_SILVER_PATH.exists())
print("SA4 Silver:", SA4_SILVER_PATH.exists())
print("EV-SA4 Silver:", EV_SA4_SILVER_PATH.exists())

DuckDB version: 1.5.5
Project root: D:\mycode\nsw-ev-data-pipeline
Database: D:\mycode\nsw-ev-data-pipeline\data\database\nsw_ev.duckdb

Silver files:
EV Silver: True
SA4 Silver: True
EV-SA4 Silver: True


## Step 2 — Create the DuckDB Database



A persistent DuckDB database is created for the analytical layer.

Unlike an in-memory database, the database file is stored on disk so that tables, views and analytical outputs can be reused across sessions.

The database will later serve as the central analytical layer between the Silver GeoParquet datasets and downstream Gold analytics.

In [5]:
con = duckdb.connect(str(DATABASE_PATH))

print("Connected to DuckDB.")
print("Database:", DATABASE_PATH)
print("Exists:", DATABASE_PATH.exists())

Connected to DuckDB.
Database: D:\mycode\nsw-ev-data-pipeline\data\database\nsw_ev.duckdb
Exists: True


## Step 3 — Enable DuckDB Spatial

The DuckDB Spatial extension is enabled to provide geospatial data types and spatial SQL functions.

Spatial functionality is required for operations involving:

- point geometries;
- polygon geometries;
- spatial relationships;
- geometry conversion;
- spatial validation and analysis.

This allows the analytical database to work directly with the geospatial datasets produced in the Silver layer.

In [6]:
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

print("DuckDB Spatial extension loaded.")

DuckDB Spatial extension loaded.


## Step 4 — Validate Spatial Functionality

A simple spatial SQL query is executed to confirm that the DuckDB Spatial extension has been loaded successfully.

A test point representing longitude and latitude coordinates is created using `ST_Point()` and converted to Well-Known Text (WKT) using `ST_AsText()`.

Successful execution confirms that spatial SQL functions are available in the current DuckDB session.

In [7]:
spatial_test = con.execute(
    """
    SELECT
        ST_AsText(
            ST_Point(151.2093, -33.8688)
        ) AS geometry
    """
).fetchone()

print("Spatial test result:")
print(spatial_test)

Spatial test result:
('POINT (151.2093 -33.8688)',)


## Step 5 — Validate Silver Dataset Availability

Before creating database tables, DuckDB reads the Silver-layer Parquet files directly.

The purpose of this step is to confirm that:

- all expected Silver datasets are accessible;
- DuckDB can read the Parquet files successfully;
- record counts remain consistent with the validated Python pipelines.

Expected record counts are:

- EV Silver: 1,958 locations;
- ABS SA4 Silver: 28 spatial regions;
- EV-SA4 Silver: 1,958 enriched locations.

In [8]:
silver_datasets = {
    "EV Silver": EV_SILVER_PATH,
    "SA4 Silver": SA4_SILVER_PATH,
    "EV-SA4 Silver": EV_SA4_SILVER_PATH,
}

for name, path in silver_datasets.items():
    row_count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet('{path.as_posix()}')
        """
    ).fetchone()[0]

    print(f"{name}: {row_count:,} rows")

EV Silver: 1,958 rows
SA4 Silver: 28 rows
EV-SA4 Silver: 1,958 rows


## Step 6 — Inspect Parquet Schemas and Spatial Types

The schemas of the spatial Silver datasets are inspected before they are persisted as DuckDB tables.

This step is particularly important for the geometry columns because GeoParquet stores geometry together with spatial metadata.

The inspection determines how DuckDB interprets:

- attribute data types;
- geometry columns;
- spatial metadata;
- enriched SA4 fields.

The resulting schema will guide the design of the persistent DuckDB analytical tables.

In [9]:
sa4_schema = con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{SA4_SILVER_PATH.as_posix()}')
    """
).df()

sa4_schema

,column_name,column_type,null,key,default,extra
0,SA4_CODE26,VARCHAR,YES,None,None,None
1,SA4_NAME26,VARCHAR,YES,None,None,None
2,GCC_CODE26,VARCHAR,YES,None,None,None
3,GCC_NAME26,VARCHAR,YES,None,None,None
4,STE_CODE26,VARCHAR,YES,None,None,None
5,STE_NAME26,VARCHAR,YES,None,None,None
6,AREASQKM26,DOUBLE,YES,None,None,None
7,geometry,GEOMETRY('EPSG:7844'),YES,None,None,None


In [10]:
ev_sa4_schema = con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{EV_SA4_SILVER_PATH.as_posix()}')
    """
).df()

ev_sa4_schema

,column_name,column_type,null,key,default,extra
0,source_object_id,BIGINT,YES,None,None,None
1,operator_raw,VARCHAR,YES,None,None,None
2,charger_type_raw,VARCHAR,YES,None,None,None
3,charger_rating_raw,VARCHAR,YES,None,None,None
4,lga_name_raw,VARCHAR,YES,None,None,None
5,postcode_raw,VARCHAR,YES,None,None,None
6,source,VARCHAR,YES,None,None,None
7,station_name,VARCHAR,YES,None,None,None
8,station_address,VARCHAR,YES,None,None,None
9,operator,VARCHAR,YES,None,None,None


In [11]:
print("SA4 geometry:")
display(
    sa4_schema[
        sa4_schema["column_name"] == "geometry"
    ]
)

print("\nEV-SA4 geometry:")
display(
    ev_sa4_schema[
        ev_sa4_schema["column_name"] == "geometry"
    ]
)

SA4 geometry:


,column_name,column_type,null,key,default,extra
7,geometry,GEOMETRY('EPSG:7844'),YES,None,None,None



EV-SA4 geometry:


,column_name,column_type,null,key,default,extra
33,geometry,GEOMETRY('EPSG:4326'),YES,None,None,None


In [12]:
important_columns = [
    "latitude",
    "longitude",
    "sa4_code",
    "sa4_name",
    "gcc_code",
    "gcc_name",
    "sa4_area_sqkm",
    "sa4_match_method",
    "sa4_match_distance_m",
    "geometry",
]

ev_sa4_schema[
    ev_sa4_schema["column_name"].isin(
        important_columns
    )
]

,column_name,column_type,null,key,default,extra
12,latitude,DOUBLE,YES,None,None,None
13,longitude,DOUBLE,YES,None,None,None
33,geometry,GEOMETRY('EPSG:4326'),YES,None,None,None
34,sa4_code,VARCHAR,YES,None,None,None
35,sa4_name,VARCHAR,YES,None,None,None
36,gcc_code,VARCHAR,YES,None,None,None
37,gcc_name,VARCHAR,YES,None,None,None
38,sa4_area_sqkm,DOUBLE,YES,None,None,None
39,sa4_match_method,VARCHAR,YES,None,None,None
40,sa4_match_distance_m,DOUBLE,YES,None,None,None


## Step 7 — Create Analytical Database Schemas


The DuckDB database is organised into logical schemas to separate processed data from downstream analytical outputs.

Two schemas are created:

- `silver` stores validated and reusable processed datasets;
- `gold` will store aggregated analytical tables and views designed for reporting and Power BI.

This separation mirrors the Bronze–Silver–Gold architecture used throughout the project.

In [13]:
con.execute(
    """
    CREATE SCHEMA IF NOT EXISTS silver;
    CREATE SCHEMA IF NOT EXISTS gold;
    """
)

schemas = con.execute(
    """
    SELECT schema_name
    FROM information_schema.schemata
    WHERE schema_name IN ('silver', 'gold')
    ORDER BY schema_name;
    """
).df()

schemas

,schema_name
0,gold
1,silver


## Step 8 — Persist Silver Datasets as DuckDB Tables


The validated Silver-layer Parquet datasets are materialised as persistent DuckDB tables.

Three tables are created:

- `silver.ev_locations` — cleaned EV charging location records;
- `silver.sa4_regions` — NSW SA4 polygon boundaries;
- `silver.ev_locations_sa4` — EV locations enriched with SA4 spatial attributes.

The tables are created directly from the validated Parquet files so that the analytical database can be rebuilt reproducibly from the Silver layer.

In [15]:
con.execute(
    f"""
    CREATE OR REPLACE TABLE silver.ev_locations AS
    SELECT *
    FROM read_parquet('{EV_SILVER_PATH.as_posix()}');
    """
)

con.execute(
    f"""
    CREATE OR REPLACE TABLE silver.sa4_regions AS
    SELECT *
    FROM read_parquet('{SA4_SILVER_PATH.as_posix()}');
    """
)

con.execute(
    f"""
    CREATE OR REPLACE TABLE silver.ev_locations_sa4 AS
    SELECT *
    FROM read_parquet('{EV_SA4_SILVER_PATH.as_posix()}');
    """
)

print("Silver tables created successfully.")

Silver tables created successfully.


## Step 9 — Validate Persistent DuckDB Tables

The persisted Silver tables are validated against the source Parquet datasets.

The validation checks:

- table availability;
- record counts;
- geometry types;
- coordinate reference systems;
- preservation of the EV–SA4 enrichment results.

This confirms that the transition from GeoParquet to DuckDB does not alter the validated Silver-layer data.

In [16]:
tables = con.execute(
    """
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_schema IN ('silver', 'gold')
    ORDER BY table_schema, table_name;
    """
).df()

tables

,table_schema,table_name
0,silver,ev_locations
1,silver,ev_locations_sa4
2,silver,sa4_regions


In [17]:
table_counts = con.execute(
    """
    SELECT
        'silver.ev_locations' AS table_name,
        COUNT(*) AS row_count
    FROM silver.ev_locations

    UNION ALL

    SELECT
        'silver.sa4_regions',
        COUNT(*)
    FROM silver.sa4_regions

    UNION ALL

    SELECT
        'silver.ev_locations_sa4',
        COUNT(*)
    FROM silver.ev_locations_sa4;
    """
).df()

table_counts

,table_name,row_count
0,silver.ev_locations,1958
1,silver.sa4_regions,28
2,silver.ev_locations_sa4,1958


In [18]:
match_validation = con.execute(
    """
    SELECT
        sa4_match_method,
        COUNT(*) AS location_count
    FROM silver.ev_locations_sa4
    GROUP BY sa4_match_method
    ORDER BY location_count DESC;
    """
).df()

match_validation

,sa4_match_method,location_count
0,within,1957
1,nearest_fallback,1


In [19]:
unmatched_count = con.execute(
    """
    SELECT COUNT(*)
    FROM silver.ev_locations_sa4
    WHERE sa4_code IS NULL;
    """
).fetchone()[0]

print("Unmatched EV locations:", unmatched_count)

assert unmatched_count == 0

Unmatched EV locations: 0


## Step 10 — Verify Persisted Spatial Types


The geometry columns are inspected again after materialisation to ensure that DuckDB preserves their spatial types and CRS metadata.

The expected spatial definitions are:

- SA4 polygons: `EPSG:7844` (GDA2020);
- EV point geometries: `EPSG:4326` (WGS 84).

In [20]:
sa4_table_schema = con.execute(
    """
    DESCRIBE silver.sa4_regions;
    """
).df()

ev_sa4_table_schema = con.execute(
    """
    DESCRIBE silver.ev_locations_sa4;
    """
).df()

print("SA4 geometry:")
display(
    sa4_table_schema[
        sa4_table_schema["column_name"] == "geometry"
    ]
)

print("\nEV-SA4 geometry:")
display(
    ev_sa4_table_schema[
        ev_sa4_table_schema["column_name"] == "geometry"
    ]
)

SA4 geometry:


,column_name,column_type,null,key,default,extra
7,geometry,GEOMETRY('EPSG:7844'),YES,None,None,None



EV-SA4 geometry:


,column_name,column_type,null,key,default,extra
33,geometry,GEOMETRY('EPSG:4326'),YES,None,None,None


In [21]:
spatial_validation = con.execute(
    """
    SELECT
        station_name,
        sa4_name,
        ST_AsText(geometry) AS point_wkt
    FROM silver.ev_locations_sa4
    LIMIT 5;
    """
).df()

spatial_validation

,station_name,sa4_name,point_wkt
0,None,Hunter Valley exc Newcastle,POINT (150.8901391 -32.26224229)
1,None,Sydney - Blacktown,POINT (150.8495966 -33.81100405)
2,None,New England and North West,POINT (151.669395 -30.5118739)
3,None,Sydney - North Sydney and Hornsby,POINT (151.167035 -33.77410115)
4,None,Richmond - Tweed,POINT (153.6136326 -28.64181886)


## Step 11 — Profile Key Analytical Fields


Before creating Gold-layer analytical tables, the key EV attributes are profiled inside DuckDB.

This step examines data completeness and category distributions for fields that may be used in downstream metrics, including:

- charger type;
- charger power;
- number of plugs;
- charging operator;
- SA4 and GCC classifications.

The profiling results are used to determine which metrics can be reliably included in the Gold analytics layer.

In [22]:
analytical_profile = con.execute(
    """
    SELECT
        COUNT(*) AS total_locations,

        COUNT(operator_standardised) AS operator_non_null,
        COUNT(charger_type) AS charger_type_non_null,
        COUNT(number_of_plugs) AS plug_count_non_null,
        COUNT(charger_power_kw) AS charger_power_non_null,

        COUNT(DISTINCT operator_standardised) AS unique_operators,
        COUNT(DISTINCT sa4_code) AS represented_sa4s,
        COUNT(DISTINCT gcc_code) AS represented_gccs

    FROM silver.ev_locations_sa4;
    """
).df()

analytical_profile

,total_locations,operator_non_null,charger_type_non_null,plug_count_non_null,charger_power_non_null,unique_operators,represented_sa4s,represented_gccs
0,1958,1958,1860,1958,1337,46,28,2


In [23]:
charger_type_distribution = con.execute(
    """
    SELECT
        charger_type,
        COUNT(*) AS location_count
    FROM silver.ev_locations_sa4
    GROUP BY charger_type
    ORDER BY location_count DESC;
    """
).df()

charger_type_distribution

,charger_type,location_count
0,AC,1427
1,DC,433
2,NaN,98


In [24]:
operator_distribution = con.execute(
    """
    SELECT
        operator_standardised,
        COUNT(*) AS location_count
    FROM silver.ev_locations_sa4
    GROUP BY operator_standardised
    ORDER BY location_count DESC
    LIMIT 15;
    """
).df()

operator_distribution

,operator_standardised,location_count
0,Exploren,301
1,Tesla,287
2,Chargefox,254
3,Non-networked,224
4,PLUS ES,150
5,EVX,111
6,NRMA,93
7,Evie,84
8,BP,60
9,JOLT,49


In [25]:
capacity_summary = con.execute(
    """
    SELECT
        MIN(number_of_plugs) AS min_plugs,
        AVG(number_of_plugs) AS avg_plugs,
        MAX(number_of_plugs) AS max_plugs,

        MIN(charger_power_kw) AS min_power_kw,
        AVG(charger_power_kw) AS avg_power_kw,
        MAX(charger_power_kw) AS max_power_kw

    FROM silver.ev_locations_sa4;
    """
).df()

capacity_summary

,min_plugs,avg_plugs,max_plugs,min_power_kw,avg_power_kw,max_power_kw
0,1,2.87334,35,3,44.002992,400


## Step 12 — Build the SA4 Gold Summary


A Gold-layer analytical table is created at SA4 level.

The table aggregates EV charging infrastructure across each NSW SA4 region and provides reusable metrics for reporting and visualisation.

The initial metrics include:

- charging location count;
- total plug count;
- average plugs per location;
- distinct operator count;
- average and maximum charger power;
- SA4 area;
- charging-location density;
- spatial fallback count.

Charging-location density is expressed as locations per 1,000 square kilometres to make regions of different geographic sizes easier to compare.

In [26]:
con.execute(
    """
    CREATE OR REPLACE TABLE gold.sa4_ev_summary AS

    SELECT
        sa4_code,
        sa4_name,
        gcc_code,
        gcc_name,

        MAX(sa4_area_sqkm) AS sa4_area_sqkm,

        COUNT(*) AS charging_location_count,

        SUM(number_of_plugs) AS total_plug_count,

        ROUND(
            AVG(number_of_plugs),
            2
        ) AS avg_plugs_per_location,

        COUNT(
            DISTINCT operator_standardised
        ) AS operator_count,

        ROUND(
            AVG(charger_power_kw),
            2
        ) AS avg_charger_power_kw,

        MAX(
            charger_power_kw
        ) AS max_charger_power_kw,

        ROUND(
            COUNT(*) * 1000.0
            / MAX(sa4_area_sqkm),
            2
        ) AS locations_per_1000_sqkm,

        SUM(
            CASE
                WHEN sa4_match_method = 'nearest_fallback'
                THEN 1
                ELSE 0
            END
        ) AS fallback_location_count

    FROM silver.ev_locations_sa4

    GROUP BY
        sa4_code,
        sa4_name,
        gcc_code,
        gcc_name;
    """
)

print("gold.sa4_ev_summary created.")

gold.sa4_ev_summary created.


In [27]:
sa4_gold = con.execute(
    """
    SELECT *
    FROM gold.sa4_ev_summary
    ORDER BY charging_location_count DESC;
    """
).df()

sa4_gold

,sa4_code,sa4_name,gcc_code,gcc_name,sa4_area_sqkm,charging_location_count,total_plug_count,avg_plugs_per_location,operator_count,avg_charger_power_kw,max_charger_power_kw,locations_per_1000_sqkm,fallback_location_count
0,118,Sydney - Eastern Suburbs,1GSYD,Greater Sydney,57.7341,221,403.0,1.82,14,24.06,75,3827.89,0.0
1,117,Sydney - City and Inner South,1GSYD,Greater Sydney,66.1170,139,388.0,2.79,20,32.88,250,2102.33,0.0
2,106,Hunter Valley exc Newcastle,1RNSW,Rest of NSW,21491.2923,128,362.0,2.83,16,30.04,185,5.96,0.0
3,101,Capital Region,1RNSW,Rest of NSW,51896.2445,117,411.0,3.51,13,50.60,250,2.25,0.0
4,120,Sydney - Inner West,1GSYD,Greater Sydney,64.5756,116,243.0,2.09,13,24.50,150,1796.34,0.0
5,103,Central West,1RNSW,Rest of NSW,70297.0605,113,269.0,2.38,19,50.17,180,1.61,0.0
6,121,Sydney - North Sydney and Hornsby,1GSYD,Greater Sydney,275.0992,109,348.0,3.19,20,42.13,175,396.22,0.0
7,111,Newcastle and Lake Macquarie,1RNSW,Rest of NSW,870.5824,87,252.0,2.90,20,33.67,350,99.93,0.0
8,112,Richmond - Tweed,1RNSW,Rest of NSW,10271.3204,76,213.0,2.80,13,46.57,250,7.40,0.0
9,114,Southern Highlands and Shoalhaven,1RNSW,Rest of NSW,6704.9619,76,203.0,2.67,18,45.20,350,11.33,0.0


In [28]:
gold_validation = con.execute(
    """
    SELECT
        COUNT(*) AS sa4_rows,
        SUM(charging_location_count) AS total_locations,
        SUM(fallback_location_count) AS total_fallbacks
    FROM gold.sa4_ev_summary;
    """
).df()

gold_validation

,sa4_rows,total_locations,total_fallbacks
0,28,1958.0,1.0


## Step 13 — Build the GCC-Level Summary



A second Gold-layer table aggregates EV charging infrastructure at the Greater Capital City Statistical Area (GCCSA) level.
For NSW, this provides a high-level comparison between:
- Greater Sydney;
- Rest of NSW.
The summary includes charging-location counts, plug capacity, operator diversity, charger power and infrastructure density.

In [29]:
con.execute(
    """
    CREATE OR REPLACE TABLE gold.gcc_ev_summary AS

    SELECT
        gcc_code,
        gcc_name,

        COUNT(*) AS charging_location_count,

        SUM(number_of_plugs) AS total_plug_count,

        ROUND(
            AVG(number_of_plugs),
            2
        ) AS avg_plugs_per_location,

        COUNT(
            DISTINCT operator_standardised
        ) AS operator_count,

        ROUND(
            AVG(charger_power_kw),
            2
        ) AS avg_charger_power_kw,

        MAX(
            charger_power_kw
        ) AS max_charger_power_kw,

        COUNT(
            DISTINCT sa4_code
        ) AS sa4_count,

        SUM(
            CASE
                WHEN sa4_match_method = 'nearest_fallback'
                THEN 1
                ELSE 0
            END
        ) AS fallback_location_count

    FROM silver.ev_locations_sa4

    GROUP BY
        gcc_code,
        gcc_name;
    """
)

gcc_gold = con.execute(
    """
    SELECT *
    FROM gold.gcc_ev_summary
    ORDER BY charging_location_count DESC;
    """
).df()

gcc_gold

,gcc_code,gcc_name,charging_location_count,total_plug_count,avg_plugs_per_location,operator_count,avg_charger_power_kw,max_charger_power_kw,sa4_count,fallback_location_count
0,1RNSW,Rest of NSW,994,2977.0,2.99,32,50.52,350,13,0.0
1,1GSYD,Greater Sydney,964,2649.0,2.75,41,40.00,400,15,1.0


In [30]:
gcc_validation = con.execute(
    """
    SELECT
        COUNT(*) AS gcc_rows,
        SUM(charging_location_count) AS total_locations,
        SUM(sa4_count) AS total_sa4s,
        SUM(fallback_location_count) AS total_fallbacks
    FROM gold.gcc_ev_summary;
    """
).df()

gcc_validation

,gcc_rows,total_locations,total_sa4s,total_fallbacks
0,2,1958.0,28.0,1.0


## Step 14 — Build the Operator Gold Summary



Charging infrastructure is also aggregated by standardised operator.

This table supports analysis of:

- operator market presence;
- geographic coverage;
- charging-location footprint;
- plug capacity;
- charger power characteristics.

The table can later be used directly in Power BI for operator rankings and coverage comparisons.

In [32]:
con.execute(
    """
    CREATE OR REPLACE TABLE gold.operator_ev_summary AS

    SELECT
        operator_standardised AS operator_name,

        COUNT(*) AS charging_location_count,

        SUM(number_of_plugs) AS total_plug_count,

        ROUND(
            AVG(number_of_plugs),
            2
        ) AS avg_plugs_per_location,

        COUNT(
            DISTINCT sa4_code
        ) AS sa4_coverage_count,

        COUNT(
            DISTINCT gcc_code
        ) AS gcc_coverage_count,

        ROUND(
            AVG(charger_power_kw),
            2
        ) AS avg_charger_power_kw,

        MAX(
            charger_power_kw
        ) AS max_charger_power_kw

    FROM silver.ev_locations_sa4

    WHERE operator_standardised IS NOT NULL

    GROUP BY
        operator_standardised;
    """
)

operator_gold = con.execute(
    """
    SELECT *
    FROM gold.operator_ev_summary
    ORDER BY charging_location_count DESC;
    """
).df()

operator_gold.head(15)

,operator_name,charging_location_count,total_plug_count,avg_plugs_per_location,sa4_coverage_count,gcc_coverage_count,avg_charger_power_kw,max_charger_power_kw
0,Exploren,301,804.0,2.67,28,2,28.17,180
1,Tesla,287,1128.0,3.93,28,2,49.70,250
2,Chargefox,254,790.0,3.11,28,2,34.89,350
3,Non-networked,224,544.0,2.43,28,2,19.50,22
4,PLUS ES,150,150.0,1.00,4,1,16.80,22
5,EVX,111,240.0,2.16,12,2,21.76,22
6,NRMA,93,305.0,3.28,23,2,67.57,180
7,Evie,84,316.0,3.76,24,2,101.49,350
8,BP,60,252.0,4.20,26,2,111.72,175
9,JOLT,49,95.0,1.94,12,1,26.02,50


In [33]:
operator_validation = con.execute(
    """
    SELECT
        COUNT(*) AS operator_rows,
        SUM(charging_location_count) AS locations_with_operator,
        MAX(sa4_coverage_count) AS max_sa4_coverage
    FROM gold.operator_ev_summary;
    """
).df()

operator_validation

,operator_rows,locations_with_operator,max_sa4_coverage
0,46,1958.0,28


## Step 15 — Build the Charger-Type Gold Summary


A charger-type summary is created to compare different categories of charging infrastructure.

The table captures the number of locations, plug capacity and charger-power characteristics associated with each charger type.

In [34]:
con.execute(
    """
    CREATE OR REPLACE TABLE gold.charger_type_summary AS

    SELECT
        COALESCE(
            charger_type,
            'Unknown'
        ) AS charger_type,

        COUNT(*) AS charging_location_count,

        SUM(number_of_plugs) AS total_plug_count,

        ROUND(
            AVG(number_of_plugs),
            2
        ) AS avg_plugs_per_location,

        ROUND(
            AVG(charger_power_kw),
            2
        ) AS avg_charger_power_kw,

        MAX(
            charger_power_kw
        ) AS max_charger_power_kw,

        COUNT(
            DISTINCT sa4_code
        ) AS sa4_coverage_count

    FROM silver.ev_locations_sa4

    GROUP BY
        COALESCE(
            charger_type,
            'Unknown'
        );
    """
)

charger_type_gold = con.execute(
    """
    SELECT *
    FROM gold.charger_type_summary
    ORDER BY charging_location_count DESC;
    """
).df()

charger_type_gold

,charger_type,charging_location_count,total_plug_count,avg_plugs_per_location,avg_charger_power_kw,max_charger_power_kw,sa4_coverage_count
0,AC,1427,3425.0,2.40,18.11,23,28
1,DC,433,1658.0,3.83,98.98,400,28
2,Unknown,98,543.0,5.54,18.25,22,27


In [35]:
gold_tables = con.execute(
    """
    SELECT
        table_name
    FROM information_schema.tables
    WHERE table_schema = 'gold'
    ORDER BY table_name;
    """
).df()

gold_tables

,table_name
0,charger_type_summary
1,gcc_ev_summary
2,operator_ev_summary
3,sa4_ev_summary


## Step 16 — Validate the Gold Analytics Layer


The Gold-layer tables are validated before they are used for downstream reporting.

The validation confirms that:

- all expected Gold tables exist;
- SA4 and GCC summaries preserve the complete EV location count;
- charger-type totals reconcile with the Silver layer;
- spatial fallback counts remain consistent;
- no unexpected duplicate geographic keys are introduced.

This ensures that the analytical layer remains consistent with the validated Silver datasets.

In [36]:
gold_validation = con.execute(
    """
    SELECT
        (SELECT COUNT(*)
         FROM gold.sa4_ev_summary) AS sa4_rows,

        (SELECT COUNT(*)
         FROM gold.gcc_ev_summary) AS gcc_rows,

        (SELECT COUNT(*)
         FROM gold.operator_ev_summary) AS operator_rows,

        (SELECT COUNT(*)
         FROM gold.charger_type_summary) AS charger_type_rows,

        (SELECT SUM(charging_location_count)
         FROM gold.sa4_ev_summary) AS sa4_total_locations,

        (SELECT SUM(charging_location_count)
         FROM gold.gcc_ev_summary) AS gcc_total_locations,

        (SELECT SUM(charging_location_count)
         FROM gold.charger_type_summary) AS charger_type_total_locations,

        (SELECT COUNT(*)
         FROM silver.ev_locations_sa4) AS silver_total_locations;
    """
).df()

gold_validation

,sa4_rows,gcc_rows,operator_rows,charger_type_rows,sa4_total_locations,gcc_total_locations,charger_type_total_locations,silver_total_locations
0,28,2,46,3,1958.0,1958.0,1958.0,1958


In [37]:
assert gold_validation["sa4_rows"].iloc[0] == 28
assert gold_validation["gcc_rows"].iloc[0] == 2
assert gold_validation["operator_rows"].iloc[0] == 46
assert gold_validation["charger_type_rows"].iloc[0] == 3

assert (
    gold_validation["sa4_total_locations"].iloc[0]
    == gold_validation["silver_total_locations"].iloc[0]
)

assert (
    gold_validation["gcc_total_locations"].iloc[0]
    == gold_validation["silver_total_locations"].iloc[0]
)

assert (
    gold_validation["charger_type_total_locations"].iloc[0]
    == gold_validation["silver_total_locations"].iloc[0]
)

duplicate_sa4 = con.execute(
    """
    SELECT COUNT(*)
    FROM (
        SELECT sa4_code
        FROM gold.sa4_ev_summary
        GROUP BY sa4_code
        HAVING COUNT(*) > 1
    );
    """
).fetchone()[0]

duplicate_gcc = con.execute(
    """
    SELECT COUNT(*)
    FROM (
        SELECT gcc_code
        FROM gold.gcc_ev_summary
        GROUP BY gcc_code
        HAVING COUNT(*) > 1
    );
    """
).fetchone()[0]

assert duplicate_sa4 == 0
assert duplicate_gcc == 0

print("Gold analytics layer validation passed.")

Gold analytics layer validation passed.


## Step 17 — Export Gold Analytics Tables


The validated Gold tables are exported to Parquet for downstream reporting and interoperability.

These files provide a lightweight interface between DuckDB and tools such as Power BI while keeping DuckDB as the reproducible analytical source of truth.

In [38]:
GOLD_DIR = PROJECT_ROOT / "data" / "gold"
GOLD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

gold_exports = {
    "sa4_ev_summary": GOLD_DIR / "sa4_ev_summary.parquet",
    "gcc_ev_summary": GOLD_DIR / "gcc_ev_summary.parquet",
    "operator_ev_summary": GOLD_DIR / "operator_ev_summary.parquet",
    "charger_type_summary": GOLD_DIR / "charger_type_summary.parquet",
}

for table_name, output_path in gold_exports.items():
    con.execute(
        f"""
        COPY gold.{table_name}
        TO '{output_path.as_posix()}'
        (FORMAT PARQUET);
        """
    )

    print(
        f"{table_name}:",
        output_path.exists(),
        output_path,
    )

sa4_ev_summary: True D:\mycode\nsw-ev-data-pipeline\data\gold\sa4_ev_summary.parquet
gcc_ev_summary: True D:\mycode\nsw-ev-data-pipeline\data\gold\gcc_ev_summary.parquet
operator_ev_summary: True D:\mycode\nsw-ev-data-pipeline\data\gold\operator_ev_summary.parquet
charger_type_summary: True D:\mycode\nsw-ev-data-pipeline\data\gold\charger_type_summary.parquet


In [39]:
for table_name, output_path in gold_exports.items():

    source_count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM gold.{table_name};
        """
    ).fetchone()[0]

    exported_count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet('{output_path.as_posix()}');
        """
    ).fetchone()[0]

    assert source_count == exported_count

    print(
        f"{table_name}: "
        f"{exported_count} rows validated"
    )

print("\nAll Gold exports validated.")

sa4_ev_summary: 28 rows validated
gcc_ev_summary: 2 rows validated
operator_ev_summary: 46 rows validated
charger_type_summary: 3 rows validated

All Gold exports validated.


In [40]:
con.close()

print("DuckDB connection closed.")
print("DuckDB and Gold analytics notebook completed successfully.")

DuckDB connection closed.
DuckDB and Gold analytics notebook completed successfully.
